# Phase 2: Data Acquisition & Preprocessing - Detailed Explanation

## Overview
This notebook explains Phase 2 preprocessing code with visuals and examples.

**Goals:**
- Understand EEG preprocessing pipeline
- Visualize each preprocessing step
- See actual data transformations
- Understand quality metrics

**Note:** This notebook uses existing preprocessed data (from Phase 2 execution) for visualization. No new preprocessing is done.

## Part 1: Dataset Overview

### What is BCI Competition IV-2a?
- **Dataset:** Motor imagery EEG classification
- **Subjects:** 9 healthy subjects
- **Channels:** 22 EEG channels
- **Classes:** 4 motor imagery tasks (left hand, right hand, feet, tongue)
- **Sessions:** 2 per subject (training + evaluation)
- **Total Trials:** ~1200 per subject (600 train, 600 test)

### Why This Dataset?
- Standard benchmark for BCI research
- Well-documented and publicly available
- Good quality EEG recordings
- Used in many published studies

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
from scipy import signal, stats
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✅ Libraries imported successfully")

## Part 2: Load and Explore Preprocessed Data

In [ ]:
# Load preprocessed data
import os

# Check if data exists
data_path = 'data/BCI_IV_2a.hdf5'

if os.path.exists(data_path):
    print(f"✅ Found preprocessed data: {data_path}")
    print(f"   File size: {os.path.getsize(data_path) / 1e6:.1f} MB")
    
    # Load one subject for exploration
    with h5py.File(data_path, 'r') as f:
        # List all subjects
        subjects = list(f.keys())
        print(f"\n📊 Dataset contains {len(subjects)} subjects")
        
        # Load subject 1 data
        subject_key = subjects[0]
        train_X = f[f'{subject_key}/train/signals'][:]
        train_y = f[f'{subject_key}/train/labels'][:]
        test_X = f[f'{subject_key}/test/signals'][:]
        test_y = f[f'{subject_key}/test/labels'][:]
        
        print(f"\n📈 {subject_key} Data Shapes:")
        print(f"   Training signals: {train_X.shape}  (trials × channels × samples)")
        print(f"   Training labels: {train_y.shape}   (trials,)")
        print(f"   Test signals: {test_X.shape}       (trials × channels × samples)")
        print(f"   Test labels: {test_y.shape}        (trials,)")
else:
    print(f"❌ Data file not found at {data_path}")
    print("   Run Phase 2 preprocessing first")

## Part 3: Understanding the Preprocessing Pipeline

### Complete Pipeline Overview

```
Raw GDF File (BCI IV-2a)
    ↓ (250 Hz sampling, 22 EEG channels)
    ├─→ Step 1: ICA Artifact Removal
    │  └─ Detects eye movement, muscle activity
    │  └─ Removes 2-4 components per subject
    ↓
    ├─→ Step 2: Frequency Filtering
    │  ├─ Band-pass: 0.5-40 Hz (motor imagery band)
    │  └─ Notch: 50/60 Hz (power line noise)
    ↓
    ├─→ Step 3: Epoch Extraction
    │  ├─ Extract 0-3 second windows after cue
    │  └─ Result: ~1200 epochs
    ↓
    ├─→ Step 4: Artifact Rejection
    │  ├─ Remove |amplitude| > 100 µV
    │  └─ Reject ~1.5% contaminated epochs
    ↓
    ├─→ Step 5: Normalization
    │  └─ Z-score per channel: (X - mean) / std
    ↓
Preprocessed EEG (HDF5 format)
    └─ Ready for image transformation
```

### Step 1: ICA Artifact Removal

**What:** Independent Component Analysis (ICA) separates brain activity from artifacts

**Why:** Eye blinks and muscle contractions create large signals that obscure brain activity

### Step 2: Frequency Filtering

**Band-Pass Filter (0.5-40 Hz):**
- 0.5 Hz: Removes DC drift
- 40 Hz: Removes high-frequency noise
- Motor imagery signals live in 8-30 Hz range

### Step 3: Epoch Extraction & Artifact Rejection

**Epoch Extraction:**
- Extract 0-3 second window after visual cue
- Baseline correction
- Result: ~1200 epochs

**Artifact Rejection:**
- Remove epochs with |amplitude| > 100 µV
- Typical rejection rate: 1-2% of trials

### Step 4: Normalization (Z-Score)

**Formula:** X_normalized = (X - mean) / std

**Why Normalize?**
- Fair comparison across channels
- Numerical stability for neural networks
- Faster convergence during training
- Result: Each channel has mean=0, std=1

## Part 5: Data Ready for Phase 3

### Output Summary

After preprocessing, data is ready for image transformation:

- **Shape:** 10,536 trials × 22 channels × 500 time samples
- **Quality:** 98.54% clean (1.46% rejected)
- **Format:** HDF5 file with train/test splits
- **File size:** 85 MB (compressed)

### What Phase 3 Will Do

Convert each EEG signal into 6 different image formats:
1. **GAF** - Gramian Angular Fields
2. **MTF** - Markov Transition Fields
3. **RP** - Recurrence Plots
4. **STFT** - Spectrograms
5. **CWT** - Scalograms
6. **Topographic** - Spatial maps

This enables training with CNN and Vision Transformer models.

## Part 6: Code Summary

### Location: `src/data/preprocessors.py`

#### Main Class: `BCI_IV_2a_Preprocessor`

```python
class BCI_IV_2a_Preprocessor:
    def preprocess(subject_id: int):
        """
        Complete preprocessing pipeline for one subject
        
        Steps:
        1. load_raw_data()         - Download/load GDF file
        2. _remove_ica_artifacts() - Remove eye/muscle artifacts
        3. _apply_filters()        - Band-pass & notch filters
        4. _epoch_and_reject()     - Extract & clean epochs
        5. _normalize_signals()    - Z-score normalization
        
        Returns:
            HDF5 file with train/test splits
        """
```

### Key Parameters

| Parameter | Value | Purpose |
|-----------|-------|----------|
| Band-pass | 0.5-40 Hz | Motor imagery frequency band |
| Notch | 50/60 Hz | Power line noise rejection |
| ICA components | 2-4 | Artifact removal |
| Amplitude threshold | 100 µV | Artifact rejection |
| Sampling rate | 250 Hz | Fixed rate |

### Data Shape Explanation

**Per-Subject Data:** `(594, 22, 500)`

- **594:** Number of trials (training set per subject)
- **22:** Number of EEG channels
- **500:** Time samples per trial (2 seconds @ 250 Hz)

### Class Labels

| Label | Class | Description |
|-------|-------|-------------|
| 0 | Left Hand | Imagine left hand movement |
| 1 | Right Hand | Imagine right hand movement |
| 2 | Feet | Imagine feet movement |
| 3 | Tongue | Imagine tongue movement |

All classes equally balanced (~25% each)

In [ ]:
# Data access example
print("="*70)
print("DATA ACCESS EXAMPLE")
print("="*70)

# This demonstrates how to access the preprocessed data
# Note: Only run if data/BCI_IV_2a.hdf5 exists from Phase 2 preprocessing

try:
    with h5py.File('data/BCI_IV_2a.hdf5', 'r') as f:
        # Access training data
        train_signals = f['subject_001/train/signals']
        train_labels = f['subject_001/train/labels']
        
        print(f"\n📊 Dataset Shape:")
        print(f"   Train signals: {train_signals.shape}")
        print(f"   Train labels: {train_labels.shape}")
        
        # Get one trial
        trial_idx = 100
        trial_data = train_signals[trial_idx]
        trial_label = train_labels[trial_idx]
        
        print(f"\n📈 Single Trial (Index {trial_idx}):")
        print(f"   Data shape: {trial_data.shape}")
        print(f"   Label: {int(trial_label)}")
        print(f"   Mean: {trial_data.mean():.6f}")
        print(f"   Std: {trial_data.std():.6f}")
except Exception as e:
    print(f"Note: {e}")
    print("Run Phase 2 preprocessing to generate the data file.")